# This pipeline will be in place for configuring and getting all the things needed in a knowledge base for ITG

In [1]:
# clean up warnings

import warnings
warnings.filterwarnings('ignore')

In [117]:
# Setup libs

%pip install -U opensearch-py
%pip install --upgrade "boto3>=1.35.50" "botocore>=1.35.50"
%pip install -U retrying

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
# restart kernel for the installation

from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

##  Step 1 : Creating Amazon Bedrock Execution Role
### Importing the Libraries and creating bedrock agent client

In [4]:
import json
import os
import boto3
import pprint
import random
from retrying import retry

In [5]:
from utility import create_bedrock_execution_role, create_oss_policy_attach_bedrock_execution_role, create_policies_in_oss

In [ ]:
def new_session_with_token(access_key, secret_key, session_token, region="us-east-1"):
    return boto3.Session(
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key,
        aws_session_token=session_token,
        region_name=region
    )

boto3_session = new_session_with_token(
    access_key="",
    secret_key="",
    session_token=""
)

boto3_session.client("sts").get_caller_identity()

{'UserId': 'AROARQ2AAR4FC62QE5GJI:sgil@itglue.com',
 'Account': '104824082186',
 'Arn': 'arn:aws:sts::104824082186:assumed-role/AWSReservedSSO_GeneralAdministratorAccess_143e80dc68dc5f36/sgil@itglue.com',
 'ResponseMetadata': {'RequestId': '5efa1226-1812-4e0d-a713-4912f1e1e720',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '5efa1226-1812-4e0d-a713-4912f1e1e720',
   'x-amz-sts-extended-request-id': 'MTp1cy1lYXN0LTE6MTc1NjEzMDA2ODY1OTpSOnJDOTBqQ29s',
   'content-type': 'text/xml',
   'content-length': '495',
   'date': 'Mon, 25 Aug 2025 13:54:28 GMT'},
  'RetryAttempts': 0}}

In [82]:
suffix = 265

boto3_session = boto3.session.Session(region_name='us-east-1')
region_name = boto3_session.region_name



In [93]:
bedrock_agent_client = boto3_session.client('bedrock-agent', region_name=region_name)

service = 'aoss'
bucket_name = "itg-knowledge-base-accounts" # replace it with your bucket name.
pp = pprint.PrettyPrinter(indent=2)

# Create the execution roles

In [136]:
import json
import random
import pprint
pp = pprint.PrettyPrinter(indent=2)

suffix = 265
# boto3_session = boto3.session.Session()
region_name = boto3_session.region_name
iam_client = boto3_session.client('iam')
account_number = boto3_session.client('sts').get_caller_identity().get('Account')
identity = boto3_session.client('sts').get_caller_identity()['Arn']

encryption_policy_name = f"bedrock-sample-rag-sp-{suffix}"
network_policy_name = f"bedrock-sample-rag-np-{suffix}"
access_policy_name = f'bedrock-sample-rag-ap-{suffix}'
bedrock_execution_role_name = f'AmazonBedrockExecutionRoleForKnowledgeBase_{suffix}'
fm_policy_name = f'AmazonBedrockFoundationModelPolicyForKnowledgeBase_{suffix}'
s3_policy_name = f'AmazonBedrockS3PolicyForKnowledgeBase_{suffix}'
oss_policy_name = f'AmazonBedrockOSSPolicyForKnowledgeBase_{suffix}'

def get_role_if_exists(role_name):
    try:
        response = iam_client.get_role(RoleName=role_name)
        return response   # existing role dict
    except ClientError as e:
        if e.response["Error"]["Code"] == "NoSuchEntity":
            return None
        raise   # re-raise unexpected errors

def create_bedrock_execution_role(bucket_name):

    # check first
    existing = get_role_if_exists(bedrock_execution_role_name)
    if existing:
        print(f"Role already exists: {existing['Role']['Arn']}")
        return existing

    foundation_model_policy_document = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": [
                    "bedrock:InvokeModel",
                ],
                "Resource": [
                    f"arn:aws:bedrock:{region_name}::foundation-model/amazon.titan-embed-text-v1"
                ]
            }
        ]
    }

    s3_policy_document = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": [
                    "s3:GetObject",
                    "s3:ListBucket"
                ],
                "Resource": [
                    f"arn:aws:s3:::{bucket_name}",
                    f"arn:aws:s3:::{bucket_name}/*"
                ],
                "Condition": {
                    "StringEquals": {
                        "aws:ResourceAccount": f"{account_number}"
                    }
                }
            }
        ]
    }

    assume_role_policy_document = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {
                    "Service": "bedrock.amazonaws.com"
                },
                "Action": "sts:AssumeRole"
            }
        ]
    }
    # create policies based on the policy documents
    pp.pprint(foundation_model_policy_document)
    fm_policy = iam_client.create_policy(
        PolicyName=fm_policy_name,
        PolicyDocument=json.dumps(foundation_model_policy_document),
        Description='Policy for accessing foundation model',
    )

    s3_policy = iam_client.create_policy(
        PolicyName=s3_policy_name,
        PolicyDocument=json.dumps(s3_policy_document),
        Description='Policy for reading documents from s3')

    # create bedrock execution role
    bedrock_kb_execution_role = iam_client.create_role(
        RoleName=bedrock_execution_role_name,
        AssumeRolePolicyDocument=json.dumps(assume_role_policy_document),
        Description='Amazon Bedrock Knowledge Base Execution Role for accessing OSS and S3',
        MaxSessionDuration=3600
    )

    # fetch arn of the policies and role created above
    bedrock_kb_execution_role_arn = bedrock_kb_execution_role['Role']['Arn']
    s3_policy_arn = s3_policy["Policy"]["Arn"]
    fm_policy_arn = fm_policy["Policy"]["Arn"]

    # attach policies to Amazon Bedrock execution role
    iam_client.attach_role_policy(
        RoleName=bedrock_kb_execution_role["Role"]["RoleName"],
        PolicyArn=fm_policy_arn
    )
    iam_client.attach_role_policy(
        RoleName=bedrock_kb_execution_role["Role"]["RoleName"],
        PolicyArn=s3_policy_arn
    )
    return bedrock_kb_execution_role


def create_oss_policy_attach_bedrock_execution_role(collection_id, bedrock_kb_execution_role):
    # define oss policy document
    oss_policy_document = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": [
                    "aoss:APIAccessAll"
                ],
                "Resource": [
                    f"arn:aws:aoss:{region_name}:{account_number}:collection/{collection_id}"
                ]
            }
        ]
    }
    oss_policy = iam_client.create_policy(
        PolicyName=oss_policy_name,
        PolicyDocument=json.dumps(oss_policy_document),
        Description='Policy for accessing opensearch serverless',
    )
    oss_policy_arn = oss_policy["Policy"]["Arn"]
    print("Opensearch serverless arn: ", oss_policy_arn)

    iam_client.attach_role_policy(
        RoleName=bedrock_kb_execution_role["Role"]["RoleName"],
        PolicyArn=oss_policy_arn
    )
    return None


def create_policies_in_oss(vector_store_name, aoss_client, bedrock_kb_execution_role_arn):
    encryption_policy = aoss_client.create_security_policy(
        name=encryption_policy_name,
        policy=json.dumps(
            {
                'Rules': [{'Resource': ['collection/' + vector_store_name],
                           'ResourceType': 'collection'}],
                'AWSOwnedKey': True
            }),
        type='encryption'
    )

    network_policy = aoss_client.create_security_policy(
        name=network_policy_name,
        policy=json.dumps(
            [
                {'Rules': [{'Resource': ['collection/' + vector_store_name],
                            'ResourceType': 'collection'}],
                 'AllowFromPublic': True}
            ]),
        type='network'
    )
    access_policy = aoss_client.create_access_policy(
        name=access_policy_name,
        policy=json.dumps(
            [
                {
                    'Rules': [
                        {
                            'Resource': ['collection/' + vector_store_name],
                            'Permission': [
                                'aoss:CreateCollectionItems',
                                'aoss:DeleteCollectionItems',
                                'aoss:UpdateCollectionItems',
                                'aoss:DescribeCollectionItems'],
                            'ResourceType': 'collection'
                        },
                        {
                            'Resource': ['index/' + vector_store_name + '/*'],
                            'Permission': [
                                'aoss:CreateIndex',
                                'aoss:DeleteIndex',
                                'aoss:UpdateIndex',
                                'aoss:DescribeIndex',
                                'aoss:ReadDocument',
                                'aoss:WriteDocument'],
                            'ResourceType': 'index'
                        }],
                    'Principal': [identity, bedrock_kb_execution_role_arn],
                    'Description': 'Easy data policy'}
            ]),
        type='data'
    )
    return encryption_policy, network_policy, access_policy


def delete_iam_role_and_policies():
    fm_policy_arn = f"arn:aws:iam::{account_number}:policy/{fm_policy_name}"
    s3_policy_arn = f"arn:aws:iam::{account_number}:policy/{s3_policy_name}"
    oss_policy_arn = f"arn:aws:iam::{account_number}:policy/{oss_policy_name}"
    iam_client.detach_role_policy(
        RoleName=bedrock_execution_role_name,
        PolicyArn=s3_policy_arn
    )
    iam_client.detach_role_policy(
        RoleName=bedrock_execution_role_name,
        PolicyArn=fm_policy_arn
    )
    iam_client.detach_role_policy(
        RoleName=bedrock_execution_role_name,
        PolicyArn=oss_policy_arn
    )
    iam_client.delete_role(RoleName=bedrock_execution_role_name)
    iam_client.delete_policy(PolicyArn=s3_policy_arn)
    iam_client.delete_policy(PolicyArn=fm_policy_arn)
    iam_client.delete_policy(PolicyArn=oss_policy_arn)
    return 0

In [137]:
# delete_iam_role_and_policies()

0

In [23]:
get_role_if_exists(bedrock_execution_role_name)

{'Role': {'Path': '/',
  'RoleName': 'AmazonBedrockExecutionRoleForKnowledgeBase_265',
  'RoleId': 'AROARQ2AAR4FFTNNX66HS',
  'Arn': 'arn:aws:iam::104824082186:role/AmazonBedrockExecutionRoleForKnowledgeBase_265',
  'CreateDate': datetime.datetime(2025, 8, 21, 18, 27, 54, tzinfo=tzlocal()),
  'AssumeRolePolicyDocument': {'Version': '2012-10-17',
   'Statement': [{'Effect': 'Allow',
     'Principal': {'Service': 'bedrock.amazonaws.com'},
     'Action': 'sts:AssumeRole'}]},
  'Description': 'Amazon Bedrock Knowledge Base Execution Role for accessing OSS and S3',
  'MaxSessionDuration': 3600,
  'RoleLastUsed': {'LastUsedDate': datetime.datetime(2025, 8, 21, 19, 57, 44, tzinfo=tzlocal()),
   'Region': 'us-east-1'}},
 'ResponseMetadata': {'RequestId': '9c6cff89-14c5-4e9e-8dab-d2d12b3633ef',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Fri, 22 Aug 2025 17:06:50 GMT',
   'x-amzn-requestid': '9c6cff89-14c5-4e9e-8dab-d2d12b3633ef',
   'content-type': 'text/xml',
   'content-length': '109

In [24]:
# Execution role with Bucket permission 
bedrock_kb_execution_role = create_bedrock_execution_role(bucket_name=bucket_name)


Role already exists: arn:aws:iam::104824082186:role/AmazonBedrockExecutionRoleForKnowledgeBase_265


In [25]:
bedrock_kb_execution_role

{'Role': {'Path': '/',
  'RoleName': 'AmazonBedrockExecutionRoleForKnowledgeBase_265',
  'RoleId': 'AROARQ2AAR4FFTNNX66HS',
  'Arn': 'arn:aws:iam::104824082186:role/AmazonBedrockExecutionRoleForKnowledgeBase_265',
  'CreateDate': datetime.datetime(2025, 8, 21, 18, 27, 54, tzinfo=tzlocal()),
  'AssumeRolePolicyDocument': {'Version': '2012-10-17',
   'Statement': [{'Effect': 'Allow',
     'Principal': {'Service': 'bedrock.amazonaws.com'},
     'Action': 'sts:AssumeRole'}]},
  'Description': 'Amazon Bedrock Knowledge Base Execution Role for accessing OSS and S3',
  'MaxSessionDuration': 3600,
  'RoleLastUsed': {'LastUsedDate': datetime.datetime(2025, 8, 21, 19, 57, 44, tzinfo=tzlocal()),
   'Region': 'us-east-1'}},
 'ResponseMetadata': {'RequestId': 'aef37d4e-f012-40de-8119-0ce27da6f713',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Fri, 22 Aug 2025 17:06:53 GMT',
   'x-amzn-requestid': 'aef37d4e-f012-40de-8119-0ce27da6f713',
   'content-type': 'text/xml',
   'content-length': '109

In [26]:
# Execution role for Amazon Bedrock
bedrock_kb_execution_role_arn = bedrock_kb_execution_role['Role']['Arn']

# SetUp Amazon OpenSearch Index
Instructions on how to set up an empty AOSS index and collection.
Discussion about the concept of a collection in Amazon OpenSearch Serverless as a logical grouping of indexes.

In [27]:
import boto3
import time


vector_store_name = f'bedrock-sample-rag-{suffix}'

#table name equivalent in Amazon Opensearch
index_name = f"bedrock-sample-rag-index-{suffix}"

#creating a client for Amazon Opensearch Serverless
aoss_client = boto3_session.client('opensearchserverless')


In [14]:
encryption_policy, network_policy, access_policy = create_policies_in_oss(vector_store_name=vector_store_name,
                       aoss_client=aoss_client,
                       bedrock_kb_execution_role_arn=bedrock_kb_execution_role_arn)

In [32]:
aoss_client.list_collections()

{'collectionSummaries': [{'id': '56gvq3j8ko12d6q3c0v2',
   'name': 'bedrock-sample-rag-265',
   'status': 'ACTIVE',
   'arn': 'arn:aws:aoss:us-east-1:104824082186:collection/56gvq3j8ko12d6q3c0v2'}],
 'ResponseMetadata': {'RequestId': 'ebbc2336-5c67-4946-900c-8e243def683f',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Fri, 22 Aug 2025 17:10:08 GMT',
   'content-type': 'application/x-amz-json-1.0',
   'content-length': '200',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'ebbc2336-5c67-4946-900c-8e243def683f'},
  'RetryAttempts': 0}}

In [54]:
# Creating collection. type='VECTORSEARCH' this implies that the collection is intended for vector search operations. 
# collection = aoss_client.create_collection(name=vector_store_name,type='VECTORSEARCH')
from botocore.exceptions import ClientError

def get_collection(vector_store_name: str, collection_type: str = "VECTORSEARCH"):
    # First, try to find an existing collection
    try:
        resp = aoss_client.list_collections()
        for coll in resp.get("collectionSummaries", []):
            if coll["name"] == vector_store_name:
                print(f"Collection already exists: {coll['arn']}")
                return coll
    except ClientError as e:
        raise RuntimeError(f"Error while checking collections: {e}")

def get_or_create_collection(vector_store_name: str, collection_type: str = "VECTORSEARCH"):
    existing_collection = get_collection(vector_store_name, collection_type)

    if existing_collection:
        return existing_collection

    # If not found, create a new one
    resp = aoss_client.create_collection(
        name=vector_store_name,
        type=collection_type
    )
    print(f"Created collection: {resp['createCollectionDetail']['arn']}")
    return resp

# --- Usage ---

In [67]:
collection = get_or_create_collection(vector_store_name)
collection = collection.get("createCollectionDetail", collection)
print(collection)

Collection already exists: arn:aws:aoss:us-east-1:104824082186:collection/56gvq3j8ko12d6q3c0v2
{'id': '56gvq3j8ko12d6q3c0v2', 'name': 'bedrock-sample-rag-265', 'status': 'ACTIVE', 'arn': 'arn:aws:aoss:us-east-1:104824082186:collection/56gvq3j8ko12d6q3c0v2'}


In [56]:
pp.pprint(collection)
time.sleep(10)

{ 'arn': 'arn:aws:aoss:us-east-1:104824082186:collection/56gvq3j8ko12d6q3c0v2',
  'id': '56gvq3j8ko12d6q3c0v2',
  'name': 'bedrock-sample-rag-265',
  'status': 'ACTIVE'}


# Creating the AOSS host name string

In [68]:
collection_id = collection.get("createCollectionDetail", collection).get("id")
host = collection_id + '.' + region_name + '.aoss.amazonaws.com'
print(host)

56gvq3j8ko12d6q3c0v2.us-east-1.aoss.amazonaws.com


In [62]:
# create oss policy and attach it to Bedrock execution role
create_oss_policy_attach_bedrock_execution_role(collection_id=collection_id,
                                                bedrock_kb_execution_role=bedrock_kb_execution_role)

EntityAlreadyExistsException: An error occurred (EntityAlreadyExists) when calling the CreatePolicy operation: A policy called AmazonBedrockOSSPolicyForKnowledgeBase_265 already exists. Duplicate names are not allowed.

##  Create vector index

This code snippet is designed to set up an OpenSearch index using the `opensearchpy` library in Python. It integrates with AWS services for authentication and index creation. Let's break down the steps:

1. **Import Libraries and Setup Authentication**:
   - `credentials = boto3.Session().get_credentials()`: Retrieves AWS credentials from the current session using `boto3`, an AWS SDK for Python.
   - `awsauth = AWSV4SignerAuth(credentials, region_name, service)`: Initializes the AWS V4 signer authentication, which will be used to securely connect to OpenSearch.

2. **Define Index Name and Configuration**:
   - `oss index`: 
An index in AMAZON Opensearch is a collection of documents with similar characteristics, akin to a database in a traditional relational database. It's defined by settings and mappings which specify its configuration, such as sharding and replication, and the schema for the data it stores, like data types and index-specific settings.
   - `index_name = f"bedrock-sample-index-{suffix}"`: Sets the OSS index name with a unique suffix.
   - `body_json`: Defines the JSON object for OSS index settings and mappings.
       - `settings` with `"index.knn": "true"`: Enables KNN (K-Nearest Neighbors) on the index for similarity search.
       - `Why KNN`: K-Nearest Neighbors (KNN) in OpenSearch configuration is used for similarity search, which is essential for applications requiring efficient and accurate retrieval of similar items from large datasets. By enabling KNN, OpenSearch can quickly find the "nearest" data points in a high-dimensional space, making it highly effective for tasks like recommendation systems, image search, and document retrieval, where the similarity between items is based on vector representations.
       - `mappings`: Specifies the schema of the index.
           - `vector`: A KNN vector field with a specified dimension (1536 in this case).
           - `text`: A standard text field.
           - `text-metadata`: Another text field for additional metadata.

4. **Create OpenSearch Client**:
   - `oss_client = OpenSearch(...)`: Creates an OpenSearch client instance.
       - `hosts`: Specifies the OpenSearch cluster endpoint.
       - `http_auth`: Sets the AWS authentication method.
       - `use_ssl`, `verify_certs`: Ensures secure SSL communication.
       - `connection_class`: Uses `RequestsHttpConnection` for making HTTP requests.
       - `timeout`: Sets a timeout value for the client requests.

5. **Allow Time for Data Access Rules Enforcement**:
   - `time.sleep(60)`: Waits for 60 seconds. This delay ensures that any data access rules or permissions are fully propagated and enforced before any operations are performed on the index.

This setup is typically used in scenarios where you need to create an OpenSearch index for text and vector-based search, leveraging AWS for secure and scalable data handling.


## Step 3 : Creating the OpenSearch Client and Index

In [63]:
from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
credentials = boto3.Session().get_credentials()
awsauth = auth = AWSV4SignerAuth(credentials, region_name, service)

In [64]:
index_name = f"bedrock-sample-index-{suffix}"


body_json = {
    "settings": {
        "index.knn": True,
        "index.number_of_shards": 1,
        "index.number_of_replicas": 1
    },
    "mappings": {
        "properties": {
            "vector": {
                "type": "knn_vector",
                "dimension": 1536,
                "method": {
                    "name": "hnsw",
                    "engine": "faiss",
                    "space_type": "cosinesimil",
                    "parameters": {
                        "ef_construction": 128,
                        "m": 16
                    }
                }
            },
            "text": { "type": "text" },
            "text_metadata": { "type": "keyword" }  # or "object" if you’ll store JSON
        }
    }
}
# Build the OpenSearch client
oss_client = OpenSearch(
    hosts=[{'host': host, 'port': 443}],
    http_auth=awsauth,
    use_ssl=True,
    verify_certs=True,
    connection_class=RequestsHttpConnection,
    timeout=300
)
# # It can take up to a minute for data access rules to be enforced
time.sleep(40)

In [65]:
# Creating an empty index in Amazon Opensearch using the client oss_client
response = oss_client.indices.create(index=index_name, body=json.dumps(body_json))
print('\nCreating index:')
print(response)

RequestError: RequestError(400, 'resource_already_exists_exception', 'OpenSearch exception [type=resource_already_exists_exception, reason=index [bedrock-sample-index-265/Snjj0pgBYqtkqfYy8EI1] already exists]- server : [envoy]')

# Creating S3 bucket if not existing already

In [34]:
# create s3 bucket if not created already

def create_s3_bucket(bucket_name, region="us-east-1"):
    s3_client = boto3.client("s3", region_name=region)

    try:
        if region == "us-east-1":
            # us-east-1 is special: no LocationConstraint needed
            response = s3_client.create_bucket(Bucket=bucket_name)
        else:
            response = s3_client.create_bucket(
                Bucket=bucket_name,
                CreateBucketConfiguration={"LocationConstraint": region}
            )
        print(f"Bucket '{bucket_name}' created successfully in region {region}.")
        return response
    except Exception as e:
        print(f"Error creating bucket: {e}")
        return None

# Example usage:
create_s3_bucket(bucket_name)

Bucket 'itg-knowledge-base-accounts' created successfully in region us-east-1.


{'ResponseMetadata': {'RequestId': '4JD72SDDX0E7C5FC',
  'HostId': 'HnnvHkwA5Moa5XsrDOa7e0UCa5cbhytRTpfgMK5CQgdkgcPo0fi+DLQqtwxDrDLCJ5wbDJZ9gtm8mMt9nZv16FjWeLh8QO3MaNuZ9ZckePw=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'HnnvHkwA5Moa5XsrDOa7e0UCa5cbhytRTpfgMK5CQgdkgcPo0fi+DLQqtwxDrDLCJ5wbDJZ9gtm8mMt9nZv16FjWeLh8QO3MaNuZ9ZckePw=',
   'x-amz-request-id': '4JD72SDDX0E7C5FC',
   'date': 'Thu, 21 Aug 2025 18:59:25 GMT',
   'location': '/itg-knowledge-base-accounts',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'Location': '/itg-knowledge-base-accounts'}

## Step 4 : Uploading Documents to Amazon S3

### Dataset
itglue knowledgebase in /python/notebooks/website_crawler/out_md

In [27]:
import boto3
import os

# Initialize S3 client
s3_client = boto3.client("s3")

def uploadDirectory(path, bucket_name, folder_name=''):
    for root, dirs, files in os.walk(path):
        for file in files:
            # Create full file path
            file_path = os.path.join(root, file)

            # Create the S3 key with folder name (if provided) and preserving directory structure
            s3_key = os.path.join(folder_name, os.path.relpath(file_path, start=path))

            # Upload file to S3
            s3_client.upload_file(file_path, bucket_name, s3_key)

# checks that we have an actual dir
def list_files_in_dir(path):
    if os.path.isdir(path):
        print(f"Directory exists: {path}")
        for file in os.listdir(path):
            print(file)
    else:
        print(f"Directory does not exist: {path}")

In [32]:
data_root = '../website_crawler/out_md'  # Set your local directory path
folder_name = 'itg-knowledge-base'           # Set your folder name (optional)

# list_files_in_dir(data_root)

In [35]:
uploadDirectory(data_root, bucket_name, folder_name)

## Step 5 : Establishing Amazon Bedrock Knowledge Base.
This code configures various components for setting up a knowledge base with Amazon Bedrock and OpenSearch. Let's break down each part of the configuration:

1. **OpenSearch Serverless Configuration**:
   - `opensearchServerlessConfiguration`: This dictionary configures how the knowledge base will interact with an OpenSearch serverless instance.
       - `collectionArn`: The Amazon Resource Name (ARN) of the collection in OpenSearch. It is retrieved from the `createCollectionDetail` of the collection object.
       - `vectorIndexName`: Name of the index in OpenSearch, specified by `index_name`.
       - `fieldMapping`: Defines the mapping between the knowledge base fields and the OpenSearch fields. It includes the vector field for embeddings, a text field for textual data, and a metadata field.

In [69]:
opensearchServerlessConfiguration = {
            "collectionArn": collection['arn'],
            "vectorIndexName": index_name,
            "fieldMapping": {
                "vectorField": "vector",
                "textField": "text",
                "metadataField": "text-metadata"
            }
        }


2. **Chunking Strategy Configuration**:
   - `chunkingStrategyConfiguration`: Specifies how the text data should be chunked before being processed.
       - `chunkingStrategy`: The strategy used for chunking, set to `"FIXED_SIZE"` here.
       - `fixedSizeChunkingConfiguration`: Configuration for the fixed-size chunking, including `maxTokens` (maximum number of tokens per chunk) and `overlapPercentage` (percentage of overlap between chunks).

In [70]:
chunkingStrategyConfiguration = {
    "chunkingStrategy": "FIXED_SIZE",
    "fixedSizeChunkingConfiguration": {
        "maxTokens": 1024,
        "overlapPercentage": 10
    }
}


3. **S3 Configuration**:
   - `s3Configuration`: This configures the connection to an S3 bucket.
       - `bucketArn`: The ARN for the S3 bucket, constructed using the `bucket_name`.
       - `inclusionPrefixes`: (Commented out) If enabled, this allows filtering of data from S3 based on specific prefixes.

In [71]:
s3Configuration = {
    "bucketArn": f"arn:aws:s3:::{bucket_name}",
    # "inclusionPrefixes":["*.*"] # you can use this if you want to create a KB using data within s3 prefixes.
}


4. **Embedding Model Configuration**:
   - `embeddingModelArn`: The ARN for the Amazon Bedrock embedding model, specifying the region and model name (`amazon.titan-embed-text-v1`).

In [72]:
embeddingModelArn = f"arn:aws:bedrock:{region_name}::foundation-model/amazon.titan-embed-text-v1"



5. **Knowledge Base Details**:
   - `name`: The name assigned to the knowledge base, appended with a unique `suffix`.
   - `description`: A brief description of the knowledge base's purpose.
   - `roleArn`: The ARN of the IAM role used by the knowledge base, stored in `bedrock_kb_execution_role_arn`.

In summary, this configuration sets up the necessary parameters for creating a knowledge base that leverages Amazon Bedrock for embeddings and OpenSearch for indexing and searching, with data sourced from an S3 bucket.

In [73]:
name = f"itg-account-knowledge-base-{suffix}"
description = "ITG internal docs"
roleArn = bedrock_kb_execution_role_arn

# Create knowledgebase

In [96]:
# Create a KnowledgeBase
from retrying import retry

def get_knowledge_base_if_exists(name: str):
    """Return knowledge base dict if it exists by name, else None."""
    paginator = bedrock_agent_client.get_paginator("list_knowledge_bases")
    for page in paginator.paginate():
        for kb in page.get("knowledgeBaseSummaries", []):
            if kb["name"] == name:
                # fetch full detail
                resp = bedrock_agent_client.get_knowledge_base(knowledgeBaseId=kb["knowledgeBaseId"])
                return resp["knowledgeBase"]
    return None

@retry(wait_random_min=1000, wait_random_max=2000,stop_max_attempt_number=7)
def create_knowledge_base_func():
    # check first
    existing = get_knowledge_base_if_exists(name)
    if existing:
        print(f"Knowledge base already exists: {existing['knowledgeBaseId']}")
        return existing

    create_kb_response = bedrock_agent_client.create_knowledge_base(
        name = name,
        description = description,
        roleArn = roleArn,
        knowledgeBaseConfiguration = {
            "type": "VECTOR",
            "vectorKnowledgeBaseConfiguration": {
                "embeddingModelArn": embeddingModelArn
            }
        },
        storageConfiguration = {
            "type": "OPENSEARCH_SERVERLESS",
            "opensearchServerlessConfiguration":opensearchServerlessConfiguration
        }
    )
    return create_kb_response["knowledgeBase"]

In [94]:
boto3_session.client("sts").get_caller_identity()

{'UserId': 'AROARQ2AAR4FC62QE5GJI:sgil@itglue.com',
 'Account': '104824082186',
 'Arn': 'arn:aws:sts::104824082186:assumed-role/AWSReservedSSO_GeneralAdministratorAccess_143e80dc68dc5f36/sgil@itglue.com',
 'ResponseMetadata': {'RequestId': '581177db-f3c1-43c8-a689-cbbfa89e8125',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '581177db-f3c1-43c8-a689-cbbfa89e8125',
   'x-amz-sts-extended-request-id': 'MTp1cy1lYXN0LTE6MTc1NTg4NTcxOTA0NTpSOk5EcGFBV3ZL',
   'content-type': 'text/xml',
   'content-length': '495',
   'date': 'Fri, 22 Aug 2025 18:01:59 GMT'},
  'RetryAttempts': 0}}

In [97]:
try:
    kb = create_knowledge_base_func()
except Exception as err:
    print(f"{err=}, {type(err)=}")

Knowledge base already exists: WVVFLP4XUG


In [98]:
pp.pprint(kb)

{ 'createdAt': datetime.datetime(2025, 8, 21, 19, 37, 56, 338813, tzinfo=tzlocal()),
  'description': 'Quantum computing Knowledgebase',
  'knowledgeBaseArn': 'arn:aws:bedrock:us-east-1:104824082186:knowledge-base/WVVFLP4XUG',
  'knowledgeBaseConfiguration': { 'type': 'VECTOR',
                                  'vectorKnowledgeBaseConfiguration': { 'embeddingModelArn': 'arn:aws:bedrock:us-east-1::foundation-model/amazon.titan-embed-text-v1'}},
  'knowledgeBaseId': 'WVVFLP4XUG',
  'name': 'itg-account-knowledge-base-265',
  'roleArn': 'arn:aws:iam::104824082186:role/AmazonBedrockExecutionRoleForKnowledgeBase_265',
  'status': 'ACTIVE',
  'storageConfiguration': { 'opensearchServerlessConfiguration': { 'collectionArn': 'arn:aws:aoss:us-east-1:104824082186:collection/56gvq3j8ko12d6q3c0v2',
                                                                   'fieldMapping': { 'metadataField': 'text-metadata',
                                                                                   

In [99]:
# Get KnowledgeBase 
get_kb_response = bedrock_agent_client.get_knowledge_base(knowledgeBaseId = kb['knowledgeBaseId'])

print(get_kb_response)

{'ResponseMetadata': {'RequestId': '88256abb-f64f-4cb7-9b42-747677a248fa', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Fri, 22 Aug 2025 18:02:42 GMT', 'content-type': 'application/json', 'content-length': '915', 'connection': 'keep-alive', 'x-amzn-requestid': '88256abb-f64f-4cb7-9b42-747677a248fa', 'x-amz-apigw-id': 'PuCOeHQMIAMEaQw=', 'x-amzn-trace-id': 'Root=1-68a8b0c2-699f36625899cae10d249d01'}, 'RetryAttempts': 0}, 'knowledgeBase': {'knowledgeBaseId': 'WVVFLP4XUG', 'name': 'itg-account-knowledge-base-265', 'knowledgeBaseArn': 'arn:aws:bedrock:us-east-1:104824082186:knowledge-base/WVVFLP4XUG', 'description': 'Quantum computing Knowledgebase', 'roleArn': 'arn:aws:iam::104824082186:role/AmazonBedrockExecutionRoleForKnowledgeBase_265', 'knowledgeBaseConfiguration': {'type': 'VECTOR', 'vectorKnowledgeBaseConfiguration': {'embeddingModelArn': 'arn:aws:bedrock:us-east-1::foundation-model/amazon.titan-embed-text-v1'}}, 'storageConfiguration': {'type': 'OPENSEARCH_SERVERLESS', 'opensear

## Step 6 : Creating and Managing Data Source in Knowledge Base

Next we need to create a data source, which will be associated with the knowledge base created above. Once the data source is ready, we can then start to ingest the documents.

- Create a DataSource in KnowledgeBase
- Get DataSource 

In [103]:
# Create a DataSource in KnowledgeBase 
# create_ds_response = bedrock_agent_client.create_data_source(
#     name = name,
#     description = description,
#     knowledgeBaseId = kb['knowledgeBaseId'],
#     dataSourceConfiguration = {
#         "type": "S3",
#         "s3Configuration":s3Configuration
#     },
#     vectorIngestionConfiguration = {
#         "chunkingConfiguration": chunkingStrategyConfiguration
#     }
# )
# ds = create_ds_response["dataSource"]
# pp.pprint(ds)

def get_data_source_if_exists(kb_id: str, ds_name: str):
    """Return a data source dict if it already exists in the KB, else None."""
    paginator = bedrock_agent_client.get_paginator("list_data_sources")
    for page in paginator.paginate(knowledgeBaseId=kb_id):
        for ds in page.get("dataSourceSummaries", []):
            if ds["name"] == ds_name:
                resp = bedrock_agent_client.get_data_source(
                    knowledgeBaseId=kb_id,
                    dataSourceId=ds["dataSourceId"]
                )
                return resp["dataSource"]
    return None

def create_data_source_func(kb: dict):
    # 1) check if exists
    existing = get_data_source_if_exists(kb["knowledgeBaseId"], name)
    if existing:
        print(f"Data source already exists: {existing['dataSourceId']}")
        return existing

    # 2) create new one
    create_ds_response = bedrock_agent_client.create_data_source(
        name=name,
        description=description,
        knowledgeBaseId=kb["knowledgeBaseId"],
        dataSourceConfiguration={
            "type": "S3",
            "s3Configuration": s3Configuration
        },
        vectorIngestionConfiguration={
            "chunkingConfiguration": chunkingStrategyConfiguration
        }
    )
    ds = create_ds_response["dataSource"]
    pp.pprint(ds)
    return ds

# --- Usage ---
# kb = create_knowledge_base_func()   # from earlier helper
ds = create_data_source_func(kb)

Data source already exists: DADNH9XQM4


In [104]:
# Get DataSource 
bedrock_agent_client.get_data_source(knowledgeBaseId = kb['knowledgeBaseId'], dataSourceId = ds["dataSourceId"])

{'ResponseMetadata': {'RequestId': '43fe1a42-b2de-4649-a48f-870640f6cc4a',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Fri, 22 Aug 2025 18:04:09 GMT',
   'content-type': 'application/json',
   'content-length': '584',
   'connection': 'keep-alive',
   'x-amzn-requestid': '43fe1a42-b2de-4649-a48f-870640f6cc4a',
   'x-amz-apigw-id': 'PuCcBFKgIAMEGFw=',
   'x-amzn-trace-id': 'Root=1-68a8b119-22dd82fe0ce06a2d44696d87'},
  'RetryAttempts': 0},
 'dataSource': {'knowledgeBaseId': 'WVVFLP4XUG',
  'dataSourceId': 'DADNH9XQM4',
  'name': 'itg-account-knowledge-base-265',
  'status': 'AVAILABLE',
  'description': 'Quantum computing Knowledgebase',
  'dataSourceConfiguration': {'type': 'S3',
   's3Configuration': {'bucketArn': 'arn:aws:s3:::itg-knowledge-base-accounts'}},
  'vectorIngestionConfiguration': {'chunkingConfiguration': {'chunkingStrategy': 'FIXED_SIZE',
    'fixedSizeChunkingConfiguration': {'maxTokens': 512,
     'overlapPercentage': 20}}},
  'dataDeletionPolicy': 'DELETE',
  

## Step 7 : Creating the Sync or Ingestion job
Once the KB and data source is created, we can start the ingestion job.
During the ingestion job, KB will fetch the documents in the data source, pre-process it to extract text, chunk it based on the chunking size provided, create embeddings of each chunk and then write it to the vector database, in this case OSS.

In [105]:
# Start an ingestion job
start_job_response = bedrock_agent_client.start_ingestion_job(knowledgeBaseId = kb['knowledgeBaseId'], dataSourceId = ds["dataSourceId"])

In [106]:
job = start_job_response["ingestionJob"]
pp.pprint(job)

{ 'dataSourceId': 'DADNH9XQM4',
  'ingestionJobId': 'XXSQSK7LPS',
  'knowledgeBaseId': 'WVVFLP4XUG',
  'startedAt': datetime.datetime(2025, 8, 22, 18, 4, 15, 964507, tzinfo=tzlocal()),
  'statistics': { 'numberOfDocumentsDeleted': 0,
                  'numberOfDocumentsFailed': 0,
                  'numberOfDocumentsScanned': 0,
                  'numberOfMetadataDocumentsModified': 0,
                  'numberOfMetadataDocumentsScanned': 0,
                  'numberOfModifiedDocumentsIndexed': 0,
                  'numberOfNewDocumentsIndexed': 0},
  'status': 'STARTING',
  'updatedAt': datetime.datetime(2025, 8, 22, 18, 4, 15, 964507, tzinfo=tzlocal())}


In [107]:
# Get job status
while(job['status']!='COMPLETE' ):
  get_job_response = bedrock_agent_client.get_ingestion_job(
      knowledgeBaseId = kb['knowledgeBaseId'],
        dataSourceId = ds["dataSourceId"],
        ingestionJobId = job["ingestionJobId"]
  )
  job = get_job_response["ingestionJob"]
pp.pprint(job)
time.sleep(10)

{ 'dataSourceId': 'DADNH9XQM4',
  'ingestionJobId': 'XXSQSK7LPS',
  'knowledgeBaseId': 'WVVFLP4XUG',
  'startedAt': datetime.datetime(2025, 8, 22, 18, 4, 15, 964507, tzinfo=tzlocal()),
  'statistics': { 'numberOfDocumentsDeleted': 0,
                  'numberOfDocumentsFailed': 0,
                  'numberOfDocumentsScanned': 106,
                  'numberOfMetadataDocumentsModified': 0,
                  'numberOfMetadataDocumentsScanned': 0,
                  'numberOfModifiedDocumentsIndexed': 0,
                  'numberOfNewDocumentsIndexed': 0},
  'status': 'COMPLETE',
  'updatedAt': datetime.datetime(2025, 8, 22, 18, 4, 40, 833836, tzinfo=tzlocal())}


In [111]:
kb_id = kb["knowledgeBaseId"]
pp.pprint(kb_id)

'WVVFLP4XUG'


In [112]:
%store kb_id

Stored 'kb_id' (str)


## Step 8 : Testing the Knowledge Base
### Using RetrieveAndGenerate API
Behind the scenes, RetrieveAndGenerate API converts queries into embeddings, searches the knowledge base, and then augments the foundation model prompt with the search results as context information and returns the FM-generated response to the question. For multi-turn conversations, Knowledge Bases manage short-term memory of the conversation to provide more contextual results.

The output of the RetrieveAndGenerate API includes the generated response, source attribution as well as the retrieved text chunks.

In [126]:
# try out KB using RetrieveAndGenerate API
bedrock_agent_runtime_client = boto3_session.client("bedrock-agent-runtime", region_name=region_name)
model_id = "us.anthropic.claude-3-7-sonnet-20250219-v1:0" 
model_arn = f'arn:aws:bedrock:us-east-1:104824082186:inference-profile/{model_id}'

In [128]:
query = """
support team is saying that they remember that they used to inform users that they require Microsoft P1 or P2 developer license to be able to rotate passwords for multitenant.
Is it still mandatory to have p1 or p2 or will it work if they have direct access, and does direct access means they have access to the tenant where they can login to admin centre using global admin creds on tenant?
"""
response = bedrock_agent_runtime_client.retrieve_and_generate(
    input={
        'text': query
    },
    retrieveAndGenerateConfiguration={
        'type': 'KNOWLEDGE_BASE',
        'knowledgeBaseConfiguration': {
            'knowledgeBaseId': kb_id,
            'modelArn': model_arn
        }
    },
)

generated_text = response['output']['text']
pp.pprint(generated_text)

('According to the information available, if you have direct access to '
 'multi-tenants, you do not need any P1 or P2 Microsoft licenses for password '
 'rotation. This is explicitly stated in the documentation.\n'
 '\n'
 "However, if you don't have direct access, you would need at least one of "
 'these licenses:\n'
 '- Microsoft 365 E5 Developer license\n'
 '- Microsoft 365 E5 Developer (without Windows and Audio Conferencing)\n'
 '- Microsoft Entra ID Governance\n'
 '- Microsoft Entra ID P2 Regarding what "direct access" means, it appears to '
 'refer to having administrative access to the tenant where you can log in to '
 'the admin center using global admin credentials. This is implied by the '
 'documentation which describes the process of accessing tenants through the '
 'Microsoft Entra ID admin center, switching between directories, and '
 'performing administrative tasks that would require global admin privileges.')


In [129]:
orchestration_template = """\
Conversation so far:
$conversation_history$

The user is asking a question related to ITGlue (documentation, configurations, passwords, assets, etc).

User query:
$query$

$output_format_instructions$

Return ONLY a concise ITGlue-focused search query string, no explanations.
"""


generation_template = """\
Conversation so far:
$conversation_history$

User query:
$query$

Context from the ITGlue knowledge base:
$search_results$

Instructions:
- Answer using ONLY the ITGlue context above (documentation, configurations, assets, credentials).
- If the ITGlue context is insufficient, say "I don't have enough information from the ITGlue knowledge base."
- Provide a clear, structured answer tailored for IT admins / technicians.
- Keep the language concise and practical, as in technical documentation.

$output_format_instructions$
"""


response = bedrock_agent_runtime_client.retrieve_and_generate(
    input={"text": query},
    retrieveAndGenerateConfiguration={
        "type": "KNOWLEDGE_BASE",
        "knowledgeBaseConfiguration": {
            "knowledgeBaseId": kb_id,
            "modelArn": model_arn,
            "orchestrationConfiguration": {
                "promptTemplate": {"textPromptTemplate": orchestration_template}
            },
            "generationConfiguration": {
                "promptTemplate": {"textPromptTemplate": generation_template}
            }
        }
    },
)

print(response["output"]["text"])

No, P1 or P2 licenses are not mandatory if you have direct access to multi-tenants. The documentation specifically states: "If you have a direct access to multi-tenants, you do not need any P1 or P2 Microsoft licenses."

If you don't have direct access, then you would need one of these licenses:
- Microsoft 365 E5 Developer license
- Microsoft 365 E5 Developer (without Windows and Audio Conferencing)
- Microsoft Entra ID Governance
- Microsoft Entra ID P2 Direct access means having administrative access to the tenant where you can log in to the admin center using global admin credentials. This is evident from the context where the documentation discusses accessing the Microsoft Entra ID admin center and managing permissions and roles directly within each tenant.

For password rotation to work properly, you need to assign specific permissions (like Directory.AccessAsUser.All, Directory.ReadWrite.All, etc.) and appropriate roles in each tenant where you want to enable password rotation.


In [119]:
def pick_model_arn(region="us-east-1", provider="ANTHROPIC", name_contains="sonnet"):
    bedrock = boto3_session.client("bedrock", region_name=region)
    resp = bedrock.list_foundation_models()
    for m in resp.get("modelSummaries", []):
        if (m.get("providerName","").upper()==provider
            and name_contains.lower() in m.get("modelId","").lower()
            and m.get("modelArn","").startswith(f"arn:aws:bedrock:{region}:")):
            return m["modelArn"], m["modelId"]
    raise RuntimeError("No matching model found in region; enable it in the console or choose a different region.")

model_arn, model_id = pick_model_arn()
print(model_arn, model_id)

arn:aws:bedrock:us-east-1::foundation-model/anthropic.claude-3-sonnet-20240229-v1:0:28k anthropic.claude-3-sonnet-20240229-v1:0:28k


In [97]:
import boto3, botocore
print(boto3.__version__, botocore.__version__)

1.33.2 1.33.13


In [121]:
import boto3

bedrock = boto3_session.client("bedrock", region_name="us-east-1")

resp = bedrock.list_foundation_models()
for m in resp["modelSummaries"]:
    print(m["modelId"], m["modelArn"])

twelvelabs.pegasus-1-2-v1:0 arn:aws:bedrock:us-east-1::foundation-model/twelvelabs.pegasus-1-2-v1:0
anthropic.claude-opus-4-1-20250805-v1:0 arn:aws:bedrock:us-east-1::foundation-model/anthropic.claude-opus-4-1-20250805-v1:0
amazon.titan-tg1-large arn:aws:bedrock:us-east-1::foundation-model/amazon.titan-tg1-large
amazon.titan-image-generator-v1:0 arn:aws:bedrock:us-east-1::foundation-model/amazon.titan-image-generator-v1:0
amazon.titan-image-generator-v1 arn:aws:bedrock:us-east-1::foundation-model/amazon.titan-image-generator-v1
amazon.titan-image-generator-v2:0 arn:aws:bedrock:us-east-1::foundation-model/amazon.titan-image-generator-v2:0
amazon.nova-premier-v1:0:8k arn:aws:bedrock:us-east-1::foundation-model/amazon.nova-premier-v1:0:8k
amazon.nova-premier-v1:0:20k arn:aws:bedrock:us-east-1::foundation-model/amazon.nova-premier-v1:0:20k
amazon.nova-premier-v1:0:1000k arn:aws:bedrock:us-east-1::foundation-model/amazon.nova-premier-v1:0:1000k
amazon.nova-premier-v1:0:mm arn:aws:bedrock:us